In [21]:
!pip install gradio

In [41]:
import os
import glob

from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import gradio as gr

# import gradio as gr

load_dotenv(override=True)

True

In [42]:
import os
print(os.getenv("API_KEY"))


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


In [43]:
print("Environment variables loaded.")

Environment variables loaded.


####rag check

In [44]:
## relaoding the vectordb without re-embedding
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5"
)

vectordb = Chroma(
    persist_directory="./vector_db",
    embedding_function=embedding
)

c:\Users\hitan\anaconda3\envs\your_env_py312\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [45]:
retreiver = vectordb.as_retriever(search_type="similarity",search_kwargs={"k": 5})

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="mistralai/devstral-2512:free",  # example
    openai_api_key=os.getenv("API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.1,
    top_p=0.9,
    max_tokens=512,
)

In [46]:
SYSTEM_PROMPT_TEMPLATE = """
You are an AI Scouting Analyst for professional VALORANT esports.

Your task is to generate concise, data-grounded scouting insights about an upcoming opponent
using ONLY the provided retrieved context from official match data.

STRICT RULES:
1. You must rely exclusively on the retrieved context.
2. Do NOT use outside knowledge, assumptions, or general VALORANT meta knowledge.
3. If the context does not contain enough information to answer a question, explicitly say:
   "Insufficient data available in the provided matches."
4. Do NOT speculate, predict outcomes, or invent tendencies.
5. Do NOT generalize beyond the scope of the provided series or maps.

OUTPUT STYLE:
- Write in clear, professional analyst language.
- Prefer bullet points over paragraphs.
- Be factual, neutral, and concise.
- Avoid hype, opinions, or narrative storytelling.

ALLOWED INSIGHTS (only if supported by context):
- Map-specific tendencies
- Agent compositions and pick patterns
- Player agent usage and consistency
- Observed attack or defense preferences
- Repeated behaviors across rounds or maps

DISALLOWED CONTENT:
- Predictions or win probabilities
- Coaching advice not supported by data
- Claims like "always", "never", or "dominant"
- Long-term trends beyond the given data
- Subjective judgments (e.g., "strong", "weak")

STRUCTURE YOUR RESPONSE AS:
- Section headers (e.g., "Map Tendencies", "Player Tendencies")
- Bullet points under each section
- A short "Key Takeaways" section with 2–3 bullets max

Remember:
Accuracy and restraint are more important than completeness.

"""

In [47]:
def answer_question(question: str, history):
    docs = retreiver.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [49]:
answer_question("Did any player switch roles or agents across maps?", [])

'**Player Role/Agent Switches Across Maps**\n\n- **Player: "SugarZ3ro"**\n  - **Map 1 (Ascent):** Played **Brimstone** (Controller).\n  - **Map 2 (Split):** Played **Omen** (Controller).\n  - **Map 3 (Haven):** Played **Viper** (Controller).\n  - *Observation:* Rotated between Controller agents across all maps.\n\n- **Player: "qw1"**\n  - **Map 1 (Ascent):** Played **Jett** (Duelist).\n  - **Map 2 (Split):** Played **Raze** (Duelist).\n  - **Map 3 (Haven):** Played **Phoenix** (Duelist).\n  - *Observation:* Switched Duelist agents on each map.\n\n- **Player: "Boo"**\n  - **Map 1 (Ascent):** Played **Sage** (Sentinel).\n  - **Map 2 (Split):** Played **Killjoy** (Sentinel).\n  - **Map 3 (Haven):** Played **Cypher** (Sentinel).\n  - *Observation:* Alternated Sentinel agents across maps.\n\n- **Player: "Rossy"**\n  - **Map 1 (Ascent):** Played **Sova** (Initiator).\n  - **Map 2 (Split):** Played **Skye** (Initiator).\n  - **Map 3 (Haven):** Played **Breach** (Initiator).\n  - *Observation:

In [48]:
gr.ChatInterface(answer_question).launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://d2ffe1e50dc08f8844.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
